# MOD-03 · NB-05 — Logistic Regression for Binary Health Outcomes
### Health Analytics with Python · Module 03: Statistical Inference
---
**Learning objectives**
- Fit and interpret logistic regression for binary outcomes (readmission, mortality)
- Report adjusted ORs with 95% CIs from model output
- Assess model fit: Hosmer-Lemeshow test, calibration plot, ROC-AUC
- Detect and handle multicollinearity (VIF)
- Build a multivariable risk score with predicted probabilities

**Estimated time:** 2.5 hours | **Level:** Intermediate → Advanced | **Libraries:** `pandas`, `numpy`, `statsmodels`, `sklearn`, `matplotlib`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, brier_score_loss,
                              average_precision_score, precision_recall_curve)
from sklearn.calibration import calibration_curve
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')
import os; os.makedirs('/tmp/mod03',exist_ok=True)

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

np.random.seed(42); N=2000
def logistic(x): return 1/(1+np.exp(-x))
base = np.random.normal(size=(N,3))
np.random.seed(99)
ages          = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes         = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers        = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
heart_failure = np.random.binomial(1,logistic(0.4*base[:,1]-1.2),N)
copd          = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
hypertension  = np.random.binomial(1,logistic(0.7*base[:,0]-0.2),N)
los_days      = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
admit_type    = np.random.choice(['Emergency','Elective','Urgent'],N,p=[0.52,0.30,0.18])
n_comorb      = diabetes+heart_failure+copd+hypertension

readmit_logit = (-3.0 + 0.7*heart_failure + 0.5*diabetes + 0.4*copd
                 + 0.02*(ages-60) + 0.3*(admit_type=='Emergency').astype(int)
                 + 0.2*(los_days>7).astype(int) + 0.3*(payers=='Medicaid').astype(int)
                 + np.random.normal(0,0.3,N))
readmitted_30 = np.random.binomial(1,logistic(readmit_logit),N)

df = pd.DataFrame({
    'patient_id':[f'PT-{i:05d}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'admit_type':admit_type,
    'los_days':los_days,'los_gt7':(los_days>7).astype(int),
    'diabetes':diabetes,'heart_failure':heart_failure,
    'copd':copd,'hypertension':hypertension,'n_comorb':n_comorb,
    'readmitted_30':readmitted_30,
})
df['age_std'] = (df['age']-df['age'].mean())/df['age'].std()

print(f"Dataset: {df.shape}")
print(f"Readmission rate: {readmitted_30.mean()*100:.1f}%  (event/non-event: {readmitted_30.sum()}/{(1-readmitted_30).sum()})")


## 2. Unadjusted (Univariable) Logistic Regression

In [ ]:
# Run univariable logistic regression for each predictor
univar_results = []
predictors = [('heart_failure','Heart failure'),('diabetes','Diabetes'),
              ('copd','COPD'),('hypertension','Hypertension'),
              ('los_gt7','LOS > 7 days'),('age','Age (per year)')]

for var, label in predictors:
    model_u = smf.logit(f'readmitted_30 ~ {var}', data=df).fit(disp=0)
    coef  = model_u.params[var]
    ci    = model_u.conf_int().loc[var]
    p     = model_u.pvalues[var]
    univar_results.append({
        'Predictor'  : label,
        'OR'         : round(np.exp(coef),2),
        'OR_lo'      : round(np.exp(ci[0]),2),
        'OR_hi'      : round(np.exp(ci[1]),2),
        'p-value'    : round(p,4),
        'Significant': p < 0.05,
    })

univ_df = pd.DataFrame(univar_results)
print("Univariable logistic regression — Adjusted OR (95% CI):")
display(univ_df.sort_values('OR',ascending=False))


## 3. Multivariable Logistic Regression

In [ ]:
formula = ('readmitted_30 ~ age_std + C(sex) '
           '+ C(payer, Treatment("Private")) '
           '+ heart_failure + diabetes + copd + hypertension + los_gt7 '
           '+ C(admit_type, Treatment("Elective"))')

model_multi = smf.logit(formula, data=df).fit(disp=0)
print(model_multi.summary())


In [ ]:
# Clean adjusted OR table
params  = model_multi.params
ci      = model_multi.conf_int()
pvals   = model_multi.pvalues

adj_or = pd.DataFrame({
    'Predictor'   : params.index,
    'log_OR'      : params.values,
    'adj_OR'      : np.exp(params.values),
    'CI_lo'       : np.exp(ci[0].values),
    'CI_hi'       : np.exp(ci[1].values),
    'p_value'     : pvals.values,
}).query("Predictor != 'Intercept'").round(3)
adj_or['Sig'] = adj_or['p_value'] < 0.05

print("Adjusted odds ratios — multivariable logistic regression:")
display(adj_or.sort_values('adj_OR',ascending=False))


## 4. Multicollinearity — Variance Inflation Factor

In [ ]:
# VIF for continuous and binary predictors
X_vif = df[['age','diabetes','heart_failure','copd','hypertension','los_gt7']].copy()
X_vif = sm.add_constant(X_vif)
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF'     : [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
}).query("Variable != 'const'").round(2)
print("Variance Inflation Factors:")
display(vif_data)
print()
print("VIF interpretation:")
print("  VIF < 5  : No meaningful collinearity")
print("  VIF 5-10 : Moderate — investigate")
print("  VIF > 10 : High — consider removing or combining predictors")


## 5. Model Evaluation — ROC, AUC, Calibration

In [ ]:
# Predicted probabilities
df['pred_prob'] = model_multi.predict(df)

# ── ROC curve ─────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(df['readmitted_30'], df['pred_prob'])
auc = roc_auc_score(df['readmitted_30'], df['pred_prob'])

fig,axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Logistic regression model evaluation', fontsize=13)

ax = axes[0]
ax.plot(fpr,tpr,color='#1F4E79',lw=2,label=f'ROC (AUC = {auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random classifier')
ax.fill_between(fpr,tpr,alpha=0.1,color='#1F4E79')
ax.set_xlabel('1 - Specificity (FPR)'); ax.set_ylabel('Sensitivity (TPR)')
ax.set_title('A  ROC Curve'); ax.legend(fontsize=9)

# Optimal threshold (Youden's J)
youden_j = tpr - fpr
opt_idx  = np.argmax(youden_j)
opt_thr  = thresholds[opt_idx]
ax.plot(fpr[opt_idx],tpr[opt_idx],'ro',ms=8,
        label=f'Optimal threshold ({opt_thr:.2f})')
ax.legend(fontsize=8)

# ── Calibration plot ──────────────────────────────────────────────────────────
ax = axes[1]
prob_true, prob_pred = calibration_curve(df['readmitted_30'], df['pred_prob'], n_bins=10)
ax.plot(prob_pred,prob_true,'o-',color='#D65F5F',lw=2,label='Model calibration')
ax.plot([0,1],[0,1],'k--',lw=1.2,label='Perfect calibration')
ax.fill_between([0,1],[0,1],[0,1],alpha=0.05,color='green')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('B  Calibration Plot (points close to diagonal = well-calibrated)')
ax.legend(fontsize=9)
brier = brier_score_loss(df['readmitted_30'],df['pred_prob'])
ax.text(0.05,0.92,f'Brier score = {brier:.3f}',transform=ax.transAxes,
        fontsize=9,bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))

# ── Precision-Recall curve ────────────────────────────────────────────────────
ax = axes[2]
precision, recall, _ = precision_recall_curve(df['readmitted_30'],df['pred_prob'])
ap = average_precision_score(df['readmitted_30'],df['pred_prob'])
ax.plot(recall,precision,color='#6ACC65',lw=2,label=f'PR curve (AP={ap:.3f})')
base_rate = df['readmitted_30'].mean()
ax.axhline(base_rate,color='red',ls='--',lw=1.2,label=f'No-skill ({base_rate:.2f})')
ax.set_xlabel('Recall (Sensitivity)'); ax.set_ylabel('Precision (PPV)')
ax.set_title('C  Precision-Recall Curve (useful for imbalanced outcomes)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/mod03/logistic_evaluation.png',bbox_inches='tight'); plt.show()

print(f"\nModel performance summary:")
print(f"  AUC-ROC     : {auc:.3f}")
print(f"  Brier score : {brier:.3f}  (lower is better; 0=perfect, 0.25=null)")
print(f"  Avg precision: {ap:.3f}")


## 6. Hosmer-Lemeshow Goodness-of-Fit

In [ ]:
def hosmer_lemeshow(y_true, y_pred, n_groups=10):
    """Hosmer-Lemeshow test for logistic regression calibration."""
    df_hl = pd.DataFrame({'y':y_true,'p':y_pred})
    df_hl['decile'] = pd.qcut(df_hl['p'],n_groups,labels=False,duplicates='drop')
    grouped = df_hl.groupby('decile')
    obs_pos = grouped['y'].sum()
    obs_neg = grouped['y'].apply(lambda x: (x==0).sum())
    exp_pos = grouped['p'].sum()
    exp_neg = grouped.apply(lambda g: (1-g['p']).sum())
    hl_stat = ((obs_pos-exp_pos)**2/exp_pos + (obs_neg-exp_neg)**2/exp_neg).sum()
    p_val   = 1 - chi2.cdf(hl_stat, df=n_groups-2)
    return hl_stat, p_val

hl_stat, hl_p = hosmer_lemeshow(df['readmitted_30'], df['pred_prob'])
print(f"Hosmer-Lemeshow test:")
print(f"  χ² = {hl_stat:.3f},  p = {hl_p:.4f}")
print()
print(f"  p > 0.05 = no evidence of poor fit (good calibration)")
print(f"  p < 0.05 = significant lack of fit (model is miscalibrated)")
print()
if hl_p > 0.05:
    print("  ✓ Model is well-calibrated by HL test.")
else:
    print("  ✗ Calibration concerns — consider recalibration or additional predictors.")


## 7. Classification at Optimal Threshold + Risk Stratification

In [ ]:
# Apply optimal threshold for classification
df['pred_class'] = (df['pred_prob'] >= opt_thr).astype(int)

# Confusion matrix
cm = confusion_matrix(df['readmitted_30'],df['pred_class'])
cm_df = pd.DataFrame(cm,
    index=pd.Index(['Actual: No','Actual: Yes'],name=''),
    columns=pd.Index(['Predicted: No','Predicted: Yes'],name=''))
print("Confusion matrix:")
display(cm_df)
print()
print("Classification report:")
print(classification_report(df['readmitted_30'],df['pred_class'],
                            target_names=['Not readmitted','Readmitted']))

# Risk stratification
df['risk_tier'] = pd.cut(df['pred_prob'],[0,0.10,0.20,0.35,1.0],
                          labels=['Low (<10%)','Moderate (10-20%)','High (20-35%)','Very high (>35%)'])
risk_summary = df.groupby('risk_tier',observed=True).agg(
    n_patients=('patient_id','count'),
    actual_readmit_rate=('readmitted_30','mean'),
    mean_pred_prob=('pred_prob','mean')
).round(3)
risk_summary['actual_readmit_rate'] = (risk_summary['actual_readmit_rate']*100).round(1)
risk_summary['mean_pred_prob']      = (risk_summary['mean_pred_prob']*100).round(1)
print("\nRisk stratification:")
display(risk_summary)


## 8. Knowledge Check
1. The adjusted OR for heart failure is 2.8. Write a one-sentence clinical interpretation.
2. AUC = 0.74. Is this a good model for clinical use? What threshold would you use?
3. The Hosmer-Lemeshow p = 0.03. What does this mean, and what would you do?
4. You add `n_comorb` to the model, but both `n_comorb` and individual comorbidity flags are included. What problem might arise?
5. Recalculate the classification table using a lower threshold (0.10). How does sensitivity/specificity change?


In [ ]:
# Exercise 5 — threshold = 0.10
df['pred_class_10'] = (df['pred_prob'] >= 0.10).astype(int)
cm10 = confusion_matrix(df['readmitted_30'],df['pred_class_10'])
print("Confusion matrix (threshold=0.10):")
print(cm10)
print(classification_report(df['readmitted_30'],df['pred_class_10'],
                             target_names=['Not readmitted','Readmitted']))
print("At threshold=0.10: higher sensitivity (catches more readmissions) but lower specificity.")


---
**Next → NB-06: Poisson & Negative Binomial Regression for Count Outcomes**